In [1]:
from pathlib import Path
from datetime import datetime
from urllib.parse import urljoin
import re
import asyncio
import pandas as pd
from playwright.async_api import async_playwright

START_YEAR = 2008
END_YEAR = 2026

NEWS_PAGE_URL = "https://www.bls.gov/ces/news.htm"

OUTPUT_DIR = Path("bls/nfp")
TEXT_DIR = OUTPUT_DIR / "text"
TEXT_DIR.mkdir(parents=True, exist_ok=True)

MANIFEST_PATH = OUTPUT_DIR / "nfp_manifest.csv"

# BLS บล็อก automated retrieval ทั้งแบบ headless Chromium
# ต้องใช้ Chrome จริง (channel="chrome") + desktop user-agent คู่กัน
UA = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/130.0.0.0 Safari/537.36"
)

ARCHIVE_URL = "https://www.bls.gov/bls/news-release/empsit.htm"

# ปี 2006-2007 ไม่มีไฟล์ .htm ใน /archives/
# ลิงก์จริงคือ /news.release/history/empsit_MMDDYYYY.txt
HISTORY_START_YEAR = 2006
HISTORY_END_YEAR = 2007

async def launch_browser(p):
    """BLS บล็อก headless Chromium (คืน 403 Access Denied)
    จึงต้องใช้ Chrome จริงผ่าน channel="chrome" """
    return await p.chromium.launch(headless=True, channel="chrome")

print("Output directory:", OUTPUT_DIR.resolve())
print(f"Target years: {START_YEAR}-{END_YEAR}")

Output directory: D:\Project_Github\gold assisstant\dataset\news\bls\nfp
Target years: 2008-2026


In [2]:
async def collect_nfp_links():
    async with async_playwright() as p:
        browser = await launch_browser(p)

        context = await browser.new_context(
            user_agent=UA,
            locale="en-US"
        )

        page = await context.new_page()

        response = await page.goto(
            NEWS_PAGE_URL,
            wait_until="domcontentloaded",
            timeout=60000
        )

        status = response.status if response else None
        print("CES News page status:", status)
        print("Page title:", await page.title())

        if status != 200:
            await browser.close()
            raise RuntimeError(
                f"เปิดหน้า CES News ไม่สำเร็จ: HTTP {status}"
            )

        anchors = await page.locator("a[href]").evaluate_all("""
            els => els.map(a => ({
                href: a.href,
                text: (a.innerText || a.textContent || '').trim()
            }))
        """)

        # ค้นหาลิงก์ Archive จากหน้าเว็บจริง
        archive_candidates = [
            item for item in anchors
            if "news-release/empsit.htm" in item["href"].lower()
        ]

        print("\nArchive candidates:")
        for item in archive_candidates:
            print(item["text"], "->", item["href"])

        if not archive_candidates:
            await browser.close()
            raise RuntimeError(
                "ไม่พบลิงก์ Archived Employment Situation "
                "บนหน้า CES News กรุณาตรวจสอบหน้าเว็บ"
            )

        archive_url = archive_candidates[0]["href"]
        print("\nArchive URL discovered:", archive_url)

        archive_response = await page.goto(
            archive_url,
            wait_until="domcontentloaded",
            timeout=60000
        )

        archive_status = (
            archive_response.status if archive_response else None
        )

        print("Archive status:", archive_status)
        print("Archive title:", await page.title())

        if archive_status != 200:
            await browser.close()
            raise RuntimeError(
                f"เปิดหน้า Archive ไม่สำเร็จ: HTTP {archive_status}. "
                "ยังไม่เริ่มดาวน์โหลดข่าว"
            )

        archive_anchors = await page.locator("a[href]").evaluate_all("""
            els => els.map(a => ({
                href: a.href,
                text: (a.innerText || a.textContent || '').trim()
            }))
        """)

        await browser.close()

    records = []
    seen_urls = set()

    for item in archive_anchors:
        url = item["href"].split("#")[0]

        # ตัวอย่าง: empsit_01072011.htm
        match = re.search(
            r"/empsit_(\d{8})\.htm$",
            url,
            flags=re.IGNORECASE
        )

        if not match:
            continue

        try:
            pub_date = datetime.strptime(match.group(1), "%m%d%Y")
        except ValueError:
            continue

        if not START_YEAR <= pub_date.year <= END_YEAR:
            continue

        if url in seen_urls:
            continue

        seen_urls.add(url)

        records.append({
            "publication_date": pub_date.strftime("%Y-%m-%d"),
            "year": pub_date.year,
            "release_url": url,
            "archive_link_text": item["text"],
            "status": "pending",
            "content_chars": 0,
            "local_file": "",
            "error": ""
        })

    manifest = pd.DataFrame(records)

    if manifest.empty:
        raise RuntimeError(
            "เปิด Archive ได้ แต่ไม่พบลิงก์ HTML รูปแบบ "
            "empsit_MMDDYYYY.htm ในช่วงปีที่กำหนด"
        )

    manifest = (
        manifest.sort_values("publication_date")
        .reset_index(drop=True)
    )

    manifest.to_csv(
        MANIFEST_PATH,
        index=False,
        encoding="utf-8-sig"
    )

    print("\nTotal HTML release links:", len(manifest))
    print(
        "Date range:",
        manifest["publication_date"].min(),
        "to",
        manifest["publication_date"].max()
    )

    display(manifest.head(10))
    display(
        manifest.groupby("year").size()
        .rename("release_count")
        .to_frame()
    )

    return manifest


manifest = await collect_nfp_links()

CES News page status: 200
Page title: Latest CES Economic News Release : U.S. Bureau of Labor Statistics

Archive candidates:
Employment Situation -> https://www.bls.gov/bls/news-release/empsit.htm

Archive URL discovered: https://www.bls.gov/bls/news-release/empsit.htm
Archive status: 200
Archive title: Employment Situation Archived News Releases : U.S. Bureau of Labor Statistics

Total HTML release links: 224
Date range: 2008-02-01 to 2026-10-02


,publication_date,year,release_url,archive_link_text,status,content_chars,local_file,error
0,2008-02-01,2008,https://www.bls.gov/news.release/archives/emps...,January 2008 Employment Situation,pending,0,,
1,2008-03-07,2008,https://www.bls.gov/news.release/archives/emps...,February 2008 Employment Situation,pending,0,,
2,2008-04-04,2008,https://www.bls.gov/news.release/archives/emps...,March 2008 Employment Situation,pending,0,,
3,2008-05-02,2008,https://www.bls.gov/news.release/archives/emps...,April 2008 Employment Situation,pending,0,,
4,2008-06-06,2008,https://www.bls.gov/news.release/archives/emps...,May 2008 Employment Situation,pending,0,,
5,2008-07-03,2008,https://www.bls.gov/news.release/archives/emps...,June 2008 Employment Situation,pending,0,,
6,2008-08-01,2008,https://www.bls.gov/news.release/archives/emps...,July 2008 Employment Situation,pending,0,,
7,2008-09-05,2008,https://www.bls.gov/news.release/archives/emps...,August 2008 Employment Situation,pending,0,,
8,2008-10-03,2008,https://www.bls.gov/news.release/archives/emps...,September 2008 Employment Situation,pending,0,,
9,2008-11-07,2008,https://www.bls.gov/news.release/archives/emps...,October 2008 Employment Situation,pending,0,,


,release_count
year,
2008,11
2009,12
2010,12
2011,12
2012,12
2013,12
2014,12
2015,12
2016,12


In [3]:
# ========================================================
# เพิ่มข่าว Employment Situation (NFP) ปี 2006-2007 (path /news.release/history/)
# ปีเหล่านี้ไม่มีไฟล์ .htm ใน /archives/
# ลิงก์จริงคือ https://www.bls.gov/news.release/history/empsit_MMDDYYYY.txt
# ========================================================
HISTORY_PATTERN = re.compile(
    r"/news\.release/history/empsit_(\d{8})\.txt$",
    flags=re.IGNORECASE,
)


async def collect_nfp_history_links():
    async with async_playwright() as p:
        browser = await launch_browser(p)

        context = await browser.new_context(
            user_agent=UA,
            locale="en-US",
        )

        page = await context.new_page()

        response = await page.goto(
            ARCHIVE_URL,
            wait_until="domcontentloaded",
            timeout=60000,
        )

        status = response.status if response else None
        print("Archive index status:", status)

        if status != 200:
            await browser.close()
            raise RuntimeError(
                f"เปิดหน้า Archive ไม่สำเร็จ: HTTP {status}"
            )

        anchors = await page.locator("a[href]").evaluate_all("""
            els => els.map(a => ({
                href: a.href,
                text: (a.innerText || a.textContent || '').trim()
            }))
        """)

        await browser.close()

    records = []
    seen_urls = {str(url) for url in manifest["release_url"]}

    for item in anchors:
        url = item["href"].split("#")[0]
        match = HISTORY_PATTERN.search(url.split("?")[0])

        if not match:
            continue

        try:
            pub_date = datetime.strptime(match.group(1), "%m%d%Y")
        except ValueError:
            continue

        if not HISTORY_START_YEAR <= pub_date.year <= HISTORY_END_YEAR:
            continue

        if url in seen_urls:
            continue

        seen_urls.add(url)

        records.append({
            "publication_date": pub_date.strftime("%Y-%m-%d"),
            "year": pub_date.year,
            "release_url": url,
            "archive_link_text": item["text"],
            "status": "pending",
            "content_chars": 0,
            "local_file": "",
            "error": ""
        })

    print(
        f"\nพบลิงก์ history "
        f"({HISTORY_START_YEAR}-{HISTORY_END_YEAR}):",
        len(records)
    )

    if not records:
        print("ไม่มีลิงก์ใหม่ให้เพิ่ม")
        return manifest

    combined = pd.concat(
        [manifest, pd.DataFrame(records)],
        ignore_index=True
    )

    before = len(combined)

    # กันซ้ำด้วย release_url แล้วเรียงตามวันเผยแพร่
    combined = (
        combined.drop_duplicates(subset="release_url", keep="first")
        .sort_values("publication_date")
        .reset_index(drop=True)
    )

    combined.to_csv(
        MANIFEST_PATH,
        index=False,
        encoding="utf-8-sig"
    )

    print("เพิ่มใหม่:", len(combined) - len(manifest))
    print("รวมทั้งหมด:", len(combined))
    print(
        "Date range:",
        combined["publication_date"].min(),
        "to",
        combined["publication_date"].max()
    )
    print("ซ้ำที่ถูกตัดออก:", before - len(combined))

    display(
        combined.groupby("year").size()
        .rename("release_count")
        .to_frame()
    )

    return combined


manifest = await collect_nfp_history_links()


Archive index status: 200

พบลิงก์ history (2006-2007): 24
เพิ่มใหม่: 24
รวมทั้งหมด: 248
Date range: 2006-01-06 to 2026-10-02
ซ้ำที่ถูกตัดออก: 0


,release_count
year,
2006,12
2007,12
2008,11
2009,12
2010,12
2011,12
2012,12
2013,12
2014,12


In [4]:
TEST_URL = (
    "https://www.bls.gov/news.release/archives/"
    "empsit_01072011.htm"
)

async def test_article(url):
    async with async_playwright() as p:
        browser = await launch_browser(p)
        context = await browser.new_context(
            user_agent=UA,
            locale="en-US",
        )
        page = await context.new_page()

        response = await page.goto(
            url,
            wait_until="domcontentloaded",
            timeout=60000
        )

        status = response.status if response else None
        title = await page.title()

        content = await page.locator("body").inner_text()

        await browser.close()

    print("URL:", url)
    print("HTTP status:", status)
    print("Title:", title)
    print("Content length:", len(content))
    print("\nPreview:\n", content[:2000])

    return status, content


test_status, test_content = await test_article(TEST_URL)

URL: https://www.bls.gov/news.release/archives/empsit_01072011.htm
HTTP status: 200
Title: Employment Situation News Release
Content length: 120891

Preview:
 Skip to Content
 

An official website of the United States government Here is how you know

 United States Department of Labor
U.S. BUREAU OF LABOR STATISTICS
Release Calendar | Subscribe
HOME 
SUBJECTS 
DATA TOOLS 
PUBLICATIONS 
ECONOMIC RELEASES 
CLASSROOM 
BETA 
Bureau of Labor Statistics
Economic News Release
Economic News Release
CES CPS PRINT:
Employment Situation News Release
Transmission of material in this release is embargoed                   USDL-11-0002
until 8:30 a.m. (EST) Friday, January 7, 2011

Technical information:
 Household data:       (202) 691-6378  *  cpsinfo@bls.gov  *  www.bls.gov/cps
 Establishment data:   (202) 691-6555  *  cesinfo@bls.gov  *  www.bls.gov/ces

Media contact:         (202) 691-5902  *  PressOffice@bls.gov


                       THE EMPLOYMENT SITUATION -- DECEMBER 2010


The unemplo

In [5]:
async def download_nfp_articles(manifest):
    async with async_playwright() as p:
        browser = await launch_browser(p)

        context = await browser.new_context(
            user_agent=UA,
            locale="en-US"
        )

        page = await context.new_page()

        for idx, row in manifest.iterrows():
            pub_date = str(row["publication_date"])
            url = str(row["release_url"])

            date_obj = datetime.strptime(pub_date, "%Y-%m-%d")
            output_path = TEXT_DIR / (
                f"empsit_{date_obj.strftime('%m%d%Y')}.txt"
            )

            # ข้ามไฟล์ที่ดาวน์โหลดสำเร็จแล้ว
            if output_path.exists() and output_path.stat().st_size > 1000:
                existing = output_path.read_text(
                    encoding="utf-8",
                    errors="replace"
                )

                if len(existing.strip()) > 1000:
                    manifest.at[idx, "status"] = "success"
                    manifest.at[idx, "content_chars"] = len(existing)
                    manifest.at[idx, "local_file"] = str(output_path)
                    manifest.at[idx, "error"] = ""
                    print(f"[{idx+1}/{len(manifest)}] SKIP {pub_date}")
                    continue

            try:
                response = await page.goto(
                    url,
                    wait_until="domcontentloaded",
                    timeout=60000
                )

                status = response.status if response else None

                if status != 200:
                    raise RuntimeError(f"HTTP {status}")

                content = await page.locator("body").inner_text()

                if len(content.strip()) < 1000:
                    raise RuntimeError(
                        f"เนื้อหาสั้นผิดปกติ: {len(content)} ตัวอักษร"
                    )

                output_path.write_text(content, encoding="utf-8")

                manifest.at[idx, "status"] = "success"
                manifest.at[idx, "content_chars"] = len(content)
                manifest.at[idx, "local_file"] = str(output_path)
                manifest.at[idx, "error"] = ""

                print(
                    f"[{idx+1}/{len(manifest)}] OK "
                    f"{pub_date} ({len(content):,} chars)"
                )

            except Exception as exc:
                manifest.at[idx, "status"] = "failed"
                manifest.at[idx, "error"] = str(exc)[:500]

                print(f"[{idx+1}/{len(manifest)}] FAILED {pub_date}: {exc}")

            # บันทึกความคืบหน้าหลังทุกรายการ
            manifest.to_csv(
                MANIFEST_PATH,
                index=False,
                encoding="utf-8-sig"
            )

            await page.wait_for_timeout(1000)

        await browser.close()

    return manifest


manifest = await download_nfp_articles(manifest)

[1/248] OK 2006-01-06 (183,833 chars)
[2/248] OK 2006-02-03 (196,598 chars)
[3/248] OK 2006-03-10 (187,424 chars)
[4/248] OK 2006-04-07 (187,662 chars)
[5/248] OK 2006-05-05 (187,061 chars)
[6/248] OK 2006-06-02 (187,207 chars)
[7/248] OK 2006-07-07 (188,368 chars)
[8/248] OK 2006-08-04 (189,134 chars)
[9/248] OK 2006-09-01 (186,915 chars)
[10/248] OK 2006-10-06 (191,417 chars)
[11/248] OK 2006-11-03 (189,182 chars)
[12/248] OK 2006-12-08 (187,786 chars)
[13/248] OK 2007-01-05 (199,542 chars)
[14/248] OK 2007-02-02 (194,656 chars)
[15/248] OK 2007-03-09 (186,809 chars)
[16/248] OK 2007-04-06 (187,865 chars)
[17/248] OK 2007-05-04 (186,080 chars)
[18/248] OK 2007-06-01 (182,727 chars)
[19/248] OK 2007-07-06 (182,215 chars)
[20/248] OK 2007-08-03 (183,139 chars)
[21/248] OK 2007-09-07 (183,030 chars)
[22/248] OK 2007-10-05 (186,602 chars)
[23/248] OK 2007-11-02 (185,780 chars)
[24/248] OK 2007-12-07 (187,613 chars)
[25/248] OK 2008-02-01 (199,329 chars)
[26/248] OK 2008-03-07 (185,115 ch

In [ ]:
# ==========================================================
# Clean dataset: BLS news release -> JSON (chunks + tables)
#   อ่านข้อความดิบจาก .../text แล้วเขียนผลลัพธ์ที่ .../clean
#   - chunks : เนื้อหาแบ่งตามหัวข้อ สำหรับ RAG (ตัดหัว/ท้ายรายงาน)
#   - tables : Table A แปลงเป็นตัวเลข (category_path + values)
#              ตารางอื่นเก็บเป็น raw table block ไว้ก่อน
#   - header : ตัดข้อความก่อนบรรทัดชื่อรายงาน (CONSUMER PRICE INDEX ...)
#   - footer : ตัดจากบรรทัด "Last Modified Date:"
#   รองรับ Table A ทุกยุคของ BLS:
#     CPI  fixed-width ไม่มี dot leader (2006-2007)
#     CPI  fixed-width มี dot leader      (2008-2022/03)
#     CPI  tab-separated                  (2022/04 เป็นต้นไป)
#     NFP  boxed ASCII "Major indicators" (2006-2010)
#     NFP  "Summary table A." tab-separated (2010 เป็นต้นไป)
# ==========================================================
import csv
import json
import os
import re
import tempfile
from pathlib import Path

from tqdm.auto import tqdm

CLEAN_KIND = "nfp"          # "cpi" หรือ "nfp"
CLEAN_DIR = OUTPUT_DIR / "clean"
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

MONTH_NUM = {
    "january": 1, "february": 2, "march": 3, "april": 4, "may": 5,
    "june": 6, "july": 7, "august": 8, "september": 9, "october": 10,
    "november": 11, "december": 12,
}
MONTH_ABBR = {
    "jan": "January", "feb": "February", "mar": "March", "apr": "April",
    "may": "May", "jun": "June", "jul": "July", "aug": "August",
    "sep": "September", "sept": "September", "oct": "October",
    "nov": "November", "dec": "December",
    "january": "January", "february": "February", "march": "March",
    "april": "April", "june": "June", "july": "July", "august": "August",
    "september": "September", "october": "October", "november": "November",
    "december": "December",
}
MONTH_FULL = {
    1: "January", 2: "February", 3: "March", 4: "April", 5: "May", 6: "June",
    7: "July", 8: "August", 9: "September", 10: "October", 11: "November",
    12: "December",
}

# ชื่อรายงานต้องขึ้นต้นบรรทัด (ยอมให้มี indent ได้) เพื่อไม่ให้ไปแมตช์ข้อความ
# ที่อ้างถึงชื่อรายงานในเนื้อหา เช่น 'news release USDL 07-0486, "The Employment
# Situation: March 2007," ...'
TITLE_RE = {
    "cpi": re.compile(
        r"^\s*CONSUMER PRICE INDEX[^A-Za-z0-9]*([A-Za-z]+)\s+(\d{4})",
        re.IGNORECASE,
    ),
    "nfp": re.compile(
        r"^\s*(?:THE\s+)?EMPLOYMENT SITUATION[^A-Za-z0-9]*([A-Za-z]+)\s+(\d{4})",
        re.IGNORECASE,
    ),
}
MONTH_TOKEN_RE = re.compile(
    r"\b(Jan|Feb|Mar|Apr|May|June|July|Jun|Jul|Aug|Sept|Sep|Oct|Nov|Dec)\.?"
)
# "Table A." / "Table A-1." / "Table 1." / "Summary table A."
# หลังจุดต้องเป็นช่องว่างหรือจบบรรทัด (ไม่ใช่ "(See Summary table A.)" ที่ขึ้นบรรทัดใหม่)
TABLE_RE = re.compile(
    r"^\s*(?P<summary>Summary\s+table|Table)\s+"
    r"(?P<label>[A-Z](?:-\d+)?\d*|\d+[A-Z]?)\.(?!\S)"
)
FOOTER_MARKER = "Last Modified Date:"
VALUE_RE = re.compile(r"-?(?:\d+\.\d+|\.\d+|\d+)")
NUMBER_RE = re.compile(r"-?[$\d][\d,]*(?:\.\d+)?")
DASH_RE = re.compile(r"^-{10,}\s*$")
STOP_LINE_RE = re.compile(r"^(?:Footnotes?\b|Notes?\b|NOTE[: ])")
PROSE_MIN_LEN = 45
MONTHLY_CHANGE_HINT = "preceding month"
# ช่องที่เป็นเส้นตกแต่งของตาราง (ขีดล่าง/ขีดกลาง/จุด ล้วน) ไม่ใช่ชื่อหมวด
RULE_CELL_RE = re.compile(r"^[_\-\s.]+$")
QUARTER_TOKENS = ("I", "II", "III", "IV")

# ----------------------------------------------------------
# ลำดับชั้นของ Table A (CPI-U, U.S. city average) สำหรับยุคที่ตารางเป็นแบบ
# tab-separated (2022/04 เป็นต้นไป) ซึ่งไฟล์ข้อความไม่มี indent ให้จับได้
# ใช้แผนผังกลุ่มหลักตามที่ BLS เผยแพร่ใน Table A
# ----------------------------------------------------------
CPI_TABLE_A_ORDER = [
    (0, "All items"),
    (1, "Food"),
    (2, "Food at home"),
    (2, "Food away from home"),
    (1, "Energy"),
    (2, "Energy commodities"),
    (3, "Gasoline (all types)"),
    (3, "Fuel oil"),
    (2, "Energy services"),
    (3, "Electricity"),
    (3, "Utility (piped) gas service"),
    (1, "All items less food and energy"),
    (2, "Commodities less food and energy commodities"),
    (3, "New vehicles"),
    (3, "Used cars and trucks"),
    (3, "Apparel"),
    (3, "Medical care commodities"),
    (2, "Services less energy services"),
    (3, "Shelter"),
    (3, "Transportation services"),
    (3, "Medical care services"),
]


def _build_path_map(order):
    stack, out = [], {}
    for lvl, name in order:
        stack = stack[:lvl]
        out[name] = list(stack)
        stack.append(name)
    return out


CPI_TABLE_A_PATHS = _build_path_map(CPI_TABLE_A_ORDER)

CONTENT_TYPE = {
    "cpi": "CPI news release",
    "nfp": "Employment Situation news release",
}


# ----------------------------------------------------------
# helpers พื้นฐาน
# ----------------------------------------------------------
def _read_lines(path):
    return Path(path).read_text(encoding="utf-8", errors="replace").splitlines()


def _find_bounds(lines, kind):
    """หา (index ชื่อรายงาน, จุดเริ่มเนื้อหา, จุดจบเนื้อหา)"""
    title_idx = None
    for i, line in enumerate(lines):
        if TITLE_RE[kind].search(line):
            title_idx = i
            break
    if title_idx is None:
        return None
    start = title_idx + 1
    end = len(lines) - 1
    for i in range(start, len(lines)):
        if lines[i].strip().startswith(FOOTER_MARKER):
            end = i - 1
            break
    while end > start and not lines[end].strip():
        end -= 1
    return title_idx, start, end


def _is_prose(line):
    """บรรทัดเนื้อเรื่อง (ไม่ใช่บรรทัดในตาราง)"""
    s = line.strip()
    if len(s) < PROSE_MIN_LEN or "\t" in s or "   " in s:
        return False
    if len(re.findall(r"[A-Za-z]{2,}", s)) < 8:
        return False
    return bool(re.search(r"[a-z]{3}", s))


def _looks_like_sentence(line):
    """ประโยคจริง ใช้ตัดจบตาราง — เข้มกว่า _is_prose เพราะชื่อหมวดยาว ๆ
    เช่น 'Frozen and refrigerated bakery products, pies, tarts, turnovers(3)'
    ก็ผ่าน _is_prose ได้เหมือนกัน"""
    if not _is_prose(line):
        return False
    s = line.strip()
    if s.endswith((")", "%", "*", ":", ",", ";")):
        return False
    if re.search(r"\.{2,}", s):          # dot leader ของตาราง
        return False
    if len(re.findall(r"\d", s)) >= 6:   # บรรทัดที่มีแต่ตัวเลข
        return False
    return bool(re.match(r"^\d+\s", s) or re.search(r"\.(?:\s|$)", s))


def _next_nonblank(lines, i, hard):
    while i <= hard and not lines[i].strip():
        i += 1
    return i if i <= hard else None


def _is_rule(text):
    """ข้อความที่เป็นเส้นตกแต่งของตาราง เช่น '________' หรือ '-----------'"""
    return bool(text) and bool(RULE_CELL_RE.fullmatch(text))


def _bare_category(name):
    """ตัดเครื่องหมายเชิงอรรถท้ายชื่อหมวด เช่น 'Food away from home(1)'"""
    return re.sub(r"\s*\(\d+\)\s*$", "", name).strip()


def _month_name(tok):
    """'Nov.' / 'November' / 'Sept.' -> 'November'"""
    t = tok.strip().lower().rstrip(".")
    if t in MONTH_ABBR:
        return MONTH_ABBR[t]
    return MONTH_ABBR.get(t[:3]) or MONTH_ABBR.get(t[:4])


def _is_paragraph_start(lines, j, hard):
    """บรรทัด j ขึ้นต้นย่อหน้าเนื้อเรื่องจริงหรือไม่
    ชื่อหมวดยาว ๆ ในตารางก็ผ่าน _is_prose ได้ แต่บรรทัดถัดจากมันคือบรรทัดว่าง + บรรทัดตัวเลข
    ส่วนย่อหน้าจริงจะมีบรรทัดเนื้อเรื่องต่อเนื่องกัน
    """
    if not _is_prose(lines[j]):
        return False
    k = _next_nonblank(lines, j + 1, hard)
    return k is None or _is_prose(lines[k])


def _starts_new_section(lines, i, hard):
    """บรรทัดนี้เป็นหัวข้อใหม่ที่เริ่มขึ้นหลังตารางหรือไม่
    ต้องเป็นบรรทัดสั้น ๆ ที่มีบรรทัดว่างคั่นก่อนหน้า และตามด้วยย่อหน้าเนื้อเรื่องจริง
    """
    if i - 1 >= 0 and lines[i - 1].strip():
        return False
    if not _heading_like(lines[i]):
        return False
    j = _next_nonblank(lines, i + 1, hard)
    return j is not None and _is_paragraph_start(lines, j, hard)


def _heading_like(line):
    # ไฟล์ยุค 2006-2008 (แปลงจาก HTML) มีหัวข้อย่อหน้า 1-3 เคาะ จึงยอมให้มี indent ได้
    indent = len(line) - len(line.lstrip())
    s = line.strip()
    if not s or indent > 3:
        return False
    if len(s) > 90 or "  " in s:
        return False
    if not re.search(r"[A-Za-z]", s):
        return False
    if TABLE_RE.match(s) or STOP_LINE_RE.match(s):
        return False
    if s[-1] in ".,;:":
        return False
    return True


def _is_heading(lines, i):
    if not _heading_like(lines[i]):
        return False
    prev_blank = i - 1 < 0 or not lines[i - 1].strip()
    nxt_blank = i + 1 >= len(lines) or not lines[i + 1].strip()
    if not (prev_blank and nxt_blank):
        return False
    j = i + 1
    while j < len(lines) and not lines[j].strip():
        j += 1
    if j >= len(lines):
        return False
    return _is_prose(lines[j]) or _heading_like(lines[j])


def _trailing_numbers(line, max_n=14):
    """ดึงค่าตัวเลขที่ต่อกันอยู่ท้ายบรรทัด -> (tokens, index ตัวแรกของตัวเลข, values)"""
    toks = [(m.group(), m.start(), m.end()) for m in re.finditer(r"\S+", line)]
    vals = []
    k = len(toks) - 1
    while k >= 0 and len(vals) < max_n:
        if VALUE_RE.fullmatch(toks[k][0]):
            vals.insert(0, toks[k][0])
            k -= 1
        else:
            break
    return toks, k, vals


def _clean_category(prefix):
    return re.sub(r"[.\s]+$", "", prefix).strip()


def _seed_pending(header_lines):
    """ดึงชื่อหมวดที่วางอยู่ระหว่างหัวคอลัมน์กับแถวข้อมูลแรก
    (ตารางแบบ tab-separated ใส่ชื่อหมวดไว้คนละบรรทัดก่อนบรรทัดตัวเลข)
    """
    last_hdr = 0
    for j, line in enumerate(header_lines, start=1):
        s = line.strip()
        if not s:
            continue
        if ("\t" in line or MONTH_TOKEN_RE.search(s)
                or re.search(r"\b(?:19|20)\d{2}\b", s)
                or re.search(r"\d", s)
                or s.startswith(("[", "(")) or s.endswith(":")
                or "categor" in s.lower()):
            last_hdr = j
    seed = []
    for line in header_lines[last_hdr:]:
        s = line.strip()
        if not s or "\t" in line or re.search(r"\d", s):
            continue
        if s.startswith(("[", "(")) or s.endswith(":"):
            continue
        seed.append(s)
    return seed


def _clean_number(tok):
    """'1,234' -> 1234.0 | 'p$16.30' -> 16.3 | '(1)' / '-' -> None"""
    t = tok.strip()
    if not t:
        return None
    t = t.replace(",", "").replace("$", "")
    t = re.sub(r"^[pPrR]", "", t).strip()
    if not t or t in {"-", "--", "---", "(1)", "NA", "n/a", "..."}:
        return None
    if not re.fullmatch(r"-?(?:\d+\.\d+|\.\d+|\d+)", t):
        return None
    return float(t)


def _table_end(lines, s, hard):
    """หาบรรทัดสุดท้ายของตาราง
    - หยุดที่ตารางถัดไป บรรทัดที่เป็นประโยคเนื้อเรื่อง หรือหัวข้อใหม่ที่เริ่มขึ้นหลังตาราง
    - เจอ Footnotes/NOTE ให้ดูว่าถัดจากบล็อกเชิงอรรถนั้นมีหัวข้อใหม่
      (เช่น Frequently Asked Questions / Technical Note) หรือไม่
      ถ้ามี = ตารางจบตรงนี้ / ถ้าไม่มี = เชิงอรรถเป็นส่วนหนึ่งของตาราง
    """
    last = s
    i = s + 1
    while i <= hard:
        line = lines[i]
        stripped = line.strip()
        if not stripped:
            i += 1
            continue
        if TABLE_RE.match(line):
            break
        if STOP_LINE_RE.match(stripped):
            j = i + 1
            while j <= hard and (not lines[j].strip()
                                 or STOP_LINE_RE.match(lines[j].strip())):
                j += 1
            if j <= hard and _heading_like(lines[j]):
                break
            last = i
            i += 1
            continue
        if (_looks_like_sentence(line)
                or _starts_new_section(lines, i, hard)):
            break
        last = i
        i += 1
    return last


def _walk_back(year, month, k):
    """k เดือนย้อนหลังจาก (year, month)"""
    seq, m, y = [], month, year
    for _ in range(k):
        seq.insert(0, (m, y))
        m -= 1
        if m == 0:
            m, y = 12, y - 1
    return seq


# ----------------------------------------------------------
# parser: ตารางแบบ tab-separated (CPI 2022+ / NFP Summary table A, B)
# ----------------------------------------------------------
def _tab_columns(header_lines, n):
    joined = "\n".join(header_lines)
    low = joined.lower()
    labels, tail = [], None

    cut = re.search(r"change from[:\s]*(.*)", joined, re.IGNORECASE | re.DOTALL)
    body = joined[: cut.start()] if cut else joined
    ended = re.search(
        r"(?:ended|ending)\s+([A-Za-z]+)\.?\s+(?:\'(\d{2})|(\d{4}))",
        body, re.IGNORECASE,
    )
    if ended:
        mon = MONTH_ABBR.get(ended.group(1).lower().rstrip("."))
        yr = ended.group(3) or ("20" + ended.group(2) if ended.group(2) else None)
        if mon and yr:
            tail = f"12_months_ended_{mon}_{yr}_unadjusted"
    if cut:
        seg = cut.group(1)
        ms = [MONTH_ABBR[x.group(1).lower().rstrip(".")]
              for x in MONTH_TOKEN_RE.finditer(seg)]
        ys = re.findall(r"\b(?:19|20)\d{2}\b", seg)
        if ms and ys:
            tail = "change_from_" + "_".join(
                f"{ms[i]}_{ys[i]}" for i in range(min(len(ms), len(ys)))
            )
        else:
            tail = "change_from_previous_period"
    if tail is None:
        tail = "value_unlabeled"

    kept = re.sub(r"change from[:\s].*", " ", body, flags=re.IGNORECASE | re.DOTALL)
    kept = re.sub(
        r"(?:ended|ending)\s+[A-Za-z]+\.?\s+(?:\'\d{2}|\d{4})", " ", kept,
        flags=re.IGNORECASE,
    )
    months = [MONTH_ABBR[m.group(1).lower().rstrip(".")]
              for m in MONTH_TOKEN_RE.finditer(kept)]
    years = re.findall(r"\b(?:19|20)\d{2}\b", kept)
    n_month = n - 1
    if len(months) >= n_month and len(years) >= n_month:
        months, years = months[:n_month], years[:n_month]
        labels = [f"{months[i]} {years[i]}" for i in range(n_month)]
        ok = True
    else:
        labels = [f"col_{i + 1}" for i in range(n_month)]
        ok = False
    labels.append(tail)
    return labels, {
        "period_labels_resolved": ok,
        "month_tokens_found": len(months),
        "year_tokens_found": len(years),
        "monthly_change_hint": MONTHLY_CHANGE_HINT in low,
    }


def _parse_tab_table(block, kind, year, month):
    first = None
    for i in range(1, len(block)):
        if "\t" not in block[i]:
            continue
        toks = [x.strip() for x in block[i].strip().split("\t")]
        nums = [_clean_number(x) for x in toks]
        if sum(1 for x in nums if x is not None) >= 3:
            first = i
            break
    if first is None:
        return None

    counts = {}
    for line in block[first:]:
        if "\t" not in line:
            continue
        toks = [x.strip() for x in line.strip().split("\t")]
        nums = [_clean_number(x) for x in toks]
        k = sum(1 for x in nums if x is not None)
        if k >= 3:
            counts[len(toks)] = counts.get(len(toks), 0) + 1
    if not counts:
        return None
    n = max(counts, key=lambda k: (counts[k], k))
    if n < 3:
        return None

    labels, col_info = _tab_columns(block[1:first], n)
    pending = _seed_pending(block[1:first])

    rows, group_stack = [], []
    skipped = 0
    for line in block[first:]:
        if "\t" in line:
            toks = [x.strip() for x in line.strip().split("\t")]
            if len(toks) != n:
                if sum(1 for x in (_clean_number(t) for t in toks)
                       if x is not None) > 0:
                    skipped += 1
                continue
            nums = [_clean_number(x) for x in toks]
            if all(x is None for x in nums):
                continue
            if len(pending) >= 2:
                group_stack = pending[:-1]
            cat = pending[-1] if pending else None
            pending = []
            if not cat:
                continue
            vals = {labels[i]: nums[i] for i in range(n) if nums[i] is not None}
            rows.append({
                "category": cat,
                "category_path": list(group_stack) + [cat],
                "hierarchy_level": len(group_stack),
                "values": vals,
                "missing_values": sum(1 for x in nums if x is None),
                "raw_row": line.rstrip(),
            })
            continue
        s = line.strip()
        if not s or STOP_LINE_RE.match(s) or TABLE_RE.match(line):
            continue
        if s.startswith(("(", "[")) or s.endswith(":"):
            continue
        pending.append(s)

    if not rows:
        return None
    if kind == "cpi":
        # ยุค tab-separated ไม่มี indent ในไฟล์ จึงใช้แผนผังกลุ่มหลักของ BLS
        mapped = 0
        for r in rows:
            anc = CPI_TABLE_A_PATHS.get(_bare_category(r["category"]))
            if anc is None:
                continue
            r["category_path"] = anc + [r["category"]]
            r["hierarchy_level"] = len(anc)
            mapped += 1
        info_extra = {
            "hierarchy_source": "static_table_a_map" if mapped else "none",
            "hierarchy_rows_mapped": mapped,
        }
    else:
        info_extra = {"hierarchy_source": "tab_groups"}

    notes = [ln.strip() for ln in block
             if re.match(r"^\s*\(?\d+\)?\s+[A-Za-z]", ln)]
    info = dict(col_info)
    info["columns"] = n
    info["rows"] = len(rows)
    info["values_parsed"] = sum(len(r["values"]) for r in rows)
    info["values_missing"] = sum(r["missing_values"] for r in rows)
    info["rows_skipped"] = skipped
    info.update(info_extra)
    return labels, rows, notes, info


# ----------------------------------------------------------
# parser: ตารางแบบ fixed-width (CPI 2006-2022/03)
# ----------------------------------------------------------
def _fixed_columns(header_lines, n, year, month):
    joined = "\n".join(header_lines).lower()
    t = 2 if (n >= 9 and "compound" in joined) else 1
    n_month = n - t
    if n_month < 1:
        return None, {}

    best_ms = []
    for line in header_lines:
        ms = [MONTH_ABBR[m.group(1).lower().rstrip(".")]
              for m in MONTH_TOKEN_RE.finditer(line)]
        if len(ms) > len(best_ms):
            best_ms = ms

    seq = _walk_back(year, month, n_month)
    expect = [MONTH_FULL[m] for m, _ in seq]
    names = best_ms[:n_month] if len(best_ms) >= n_month else None
    match = bool(names) and [x.lower()[:3] for x in names] == \
        [x.lower()[:3] for x in expect]

    labels = [f"{MONTH_FULL[m]} {y}" for m, y in seq]
    tail_m, tail_y = seq[-1]
    if t == 2:
        labels.append(
            f"compound_annual_rate_3mos_ended_{MONTH_FULL[tail_m]}_{tail_y}"
        )
    labels.append(
        f"12_months_ended_{MONTH_FULL[tail_m]}_{tail_y}_unadjusted"
    )
    return labels, {
        "periods_match_report": match,
        "month_header_found": len(best_ms),
        "tail_columns": t,
    }


def _parse_fixed_table(block, kind, year, month):
    counts = {}
    for line in block[1:]:
        if "|" in line:
            continue
        vals = _trailing_numbers(line)[2]
        if len(vals) >= 3:
            counts[len(vals)] = counts.get(len(vals), 0) + 1
    if not counts:
        return None
    n = max(counts, key=lambda k: (counts[k], k))

    labels, col_info = _fixed_columns(block[1:], n, year, month)
    if not labels:
        return None

    raw = []
    started = False
    for line in block[1:]:
        if not line.strip() or DASH_RE.match(line.strip()):
            continue
        toks, k, vals = _trailing_numbers(line)
        if "|" in line:
            continue
        if len(vals) == n:
            prefix = line[: toks[k][2]] if k >= 0 else ""
            raw.append({"indent": len(line) - len(line.lstrip()),
                        "cat": _clean_category(prefix), "values": vals})
            started = True
        elif started:
            core = _clean_category(line)
            if (core and not re.search(r"\d", core)
                    and len(re.findall(r"[A-Za-z]", core)) >= 3
                    and len(core) <= 80):
                raw.append({"indent": len(line) - len(line.lstrip()),
                            "cat": core, "values": None})
    if not raw:
        return None

    indents = sorted({e["indent"] for e in raw})
    rank = {v: i for i, v in enumerate(indents)}   # ระดับชั้นจาก indent จริง

    stack, rows, pending = [], [], []
    for e in raw:
        if e["values"] is None:
            pending.append(e)
            continue
        base = e["cat"]
        if pending and base and base[0].islower():
            head = pending[-1]
            cat = f"{head['cat']} {base}".strip()
            lvl = rank[head["indent"]]
            groups = pending[:-1]
        else:
            cat, lvl, groups = base, rank[e["indent"]], pending
        for g in groups:
            gl = rank[g["indent"]]
            stack = [x for x in stack if x[0] < gl]
            stack.append((gl, g["cat"]))
        pending = []
        stack = [x for x in stack if x[0] < lvl]
        stack.append((lvl, cat))
        rows.append({
            "category": cat,
            "category_path": [x[1] for x in stack],
            "hierarchy_level": lvl,
            "values": {labels[i]: _clean_number(v)
                       for i, v in enumerate(e["values"])},
            "missing_values": sum(1 for v in e["values"]
                                  if _clean_number(v) is None),
            "raw_row": "",
        })
    if not rows:
        return None

    notes = [ln.strip() for ln in block
             if re.match(r"^\s*\d+\s+[A-Za-z]", ln)]
    info = dict(col_info)
    info["columns"] = n
    info["rows"] = len(rows)
    info["values_parsed"] = sum(len(r["values"]) for r in rows)
    info["values_missing"] = sum(r["missing_values"] for r in rows)
    info["hierarchy_source"] = "indentation"
    return labels, rows, notes, info


# ----------------------------------------------------------
# parser: NFP Table A แบบ boxed ASCII "Major indicators ..."
# ----------------------------------------------------------
def _parse_nfp_boxed(block, kind, year, month):
    counts = {}
    for line in block:
        if "|" not in line:
            continue
        fields = line.split("|")
        nums = [_clean_number(x) for x in fields[1:]]
        if len(fields) >= 4 and sum(1 for x in nums if x is not None) >= 3:
            counts[len(fields) - 1] = counts.get(len(fields) - 1, 0) + 1
    if not counts:
        return None
    n = max(counts, key=lambda k: (counts[k], k))

    # แถวข้อมูลแถวแรก = มี dots + ตัวเลขครบ
    first = None
    for i, line in enumerate(block):
        if "|" not in line:
            continue
        fields = line.split("|")
        nums = [_clean_number(x) for x in fields[1:]]
        if (len(fields) == n + 1 and re.search(r"\.{3,}", fields[0])
                and sum(1 for x in nums if x is not None) >= 3):
            first = i
            break
    if first is None:
        return None

    header = "\n".join(block[:first])
    labels, col_info = _nfp_boxed_columns(header, n, year, month)

    raw, pending_cat, groups = [], [], []
    # กลุ่มหัวข้อที่วางอยู่ระหว่างหัวคอลัมน์กับแถวข้อมูลแรก
    # (เช่น HOUSEHOLD DATA > Labor force status)
    for line in block[1:first]:
        if "|" not in line:
            continue
        fields = line.split("|")
        if len(fields) != 2:
            continue
        g = "" if _is_rule(fields[0]) else _clean_category(fields[0])
        sub = "" if _is_rule(fields[1]) else _clean_category(fields[1])
        if g or sub:
            groups = [x for x in (g, sub) if x]

    for line in block[first:]:
        if "|" not in line:
            continue
        fields = line.split("|")
        nums = [_clean_number(x) for x in fields[1:]]
        has_num = any(x is not None for x in nums)
        if len(fields) == n + 1 and has_num:
            cat = _clean_category(fields[0])
            if pending_cat:
                cat = f"{' '.join(pending_cat)} {cat}".strip()
            raw.append({"indent": len(fields[0]) - len(fields[0].lstrip()),
                        "cat": cat, "values": nums, "groups": list(groups)})
            pending_cat = []
            continue
        if len(fields) == 2:
            g = "" if _is_rule(fields[0]) else _clean_category(fields[0])
            sub = "" if _is_rule(fields[1]) else _clean_category(fields[1])
            if g or sub:
                groups = [x for x in (g, sub) if x]
                pending_cat = []
            continue
        if len(fields) > 2 and not has_num:
            txt = "" if _is_rule(fields[0]) else _clean_category(fields[0])
            if txt:
                pending_cat.append(txt)

    if not raw:
        return None
    indents = sorted({e["indent"] for e in raw})
    rank = {v: i for i, v in enumerate(indents)}
    stack, rows = [], []
    for e in raw:
        lvl = rank[e["indent"]] + 1        # +1 เพราะมี group ระดับบนสุด
        stack = [x for x in stack if x[0] < lvl]
        stack.append((lvl, e["cat"]))
        rows.append({
            "category": e["cat"],
            "category_path": e["groups"] + [x[1] for x in stack],
            "hierarchy_level": len(e["groups"]) + lvl,
            "values": {labels[i]: e["values"][i]
                       for i in range(min(len(labels), len(e["values"])))
                       if e["values"][i] is not None},
            "missing_values": sum(1 for x in e["values"][:len(labels)]
                                  if x is None),
            "raw_row": "",
        })
    info = dict(col_info)
    info["columns"] = n
    info["rows"] = len(rows)
    info["values_parsed"] = sum(len(r["values"]) for r in rows)
    info["values_missing"] = sum(r["missing_values"] for r in rows)
    info["hierarchy_source"] = "boxed_groups"
    return labels, rows, [], info


def _nfp_boxed_columns(header, n, year, month):
    """หัวคอลัมน์ของ Table A แบบ boxed = [ไตรมาส...] [เดือน...] [change]

    - ไฟล์ปี 2010 เป็นต้นไป แยกบรรทัดปีให้แต่ละคอลัมน์ จึงอ่านปีตรง ๆ ได้
    - ไฟล์ปี 2006-2009 รวมช่องปี (2005 คร่อม 2 คอลัมน์) จึงถอยหลังจากเดือนที่รายงาน
    """
    lines = header.splitlines()
    leaf_row = max(
        lines,
        key=lambda l: len(MONTH_TOKEN_RE.findall(l))
        + len(re.findall(r"\b(?:I|II|III|IV)\b", l)),
        default="",
    )
    leaves = [c.strip() for c in leaf_row.split("|")[1:] if c.strip()]
    kinds = []
    for c in leaves:
        if MONTH_TOKEN_RE.match(c):
            kinds.append("month")
        elif c in QUARTER_TOKENS:
            kinds.append("quarter")
        else:
            kinds.append(None)
    n_month = n - 1
    mpos = [i for i, k in enumerate(kinds) if k == "month"][:n_month]
    qpos = [i for i, k in enumerate(kinds) if k == "quarter"][:n_month]

    # บรรทัดปีที่แยกให้แต่ละคอลัมน์ (ถ้ามี)
    cell_years = []
    for l in lines:
        ys = [c.strip() for c in l.split("|")[1:]
              if re.fullmatch(r"(?:19|20)\d{2}", c.strip())]
        if len(ys) >= n_month:
            cell_years = ys[:n_month]
            break

    labels = [None] * n_month
    seq = _walk_back(year, month, len(mpos))
    for k, i in enumerate(mpos):
        if i < n_month:
            y = cell_years[i] if len(cell_years) > i else str(seq[k][1])
            labels[i] = f"{MONTH_FULL[seq[k][0]]} {y}"

    # คอลัมน์ไตรมาสอยู่ก่อนคอลัมน์เดือน และสิ้นสุดที่ไตรมาสของเดือนแรก
    base_m, base_y = seq[0] if seq else (month, year)
    q_idx = (base_m - 1) // 3 + 1
    q_yr = base_y
    for i in reversed(qpos):
        if i >= n_month:
            continue
        labels[i] = f"{q_yr} Q{q_idx}"
        q_idx -= 1
        if q_idx == 0:
            q_idx, q_yr = 4, q_yr - 1

    for i in range(n_month):
        if labels[i] is None:
            labels[i] = f"col_{i + 1}"

    # หัวคอลัมน์ change = เศษข้อความท้ายบรรทัดถัดจาก "|" ตัวสุดท้าย
    # (เช่น "Nov.-" / "Dec." / "change" ที่แยกกันคนละบรรทัด)
    flat = re.sub(r"[_\s]+", " ", " ".join(l.split("|")[-1]
                                           for l in lines if "|" in l))
    m = re.search(r"([A-Za-z]{3,9})\.?\s*-\s*([A-Za-z]{3,9})\.?\s*change",
                  flat, re.IGNORECASE)
    tail = "change"
    if m:
        a, b = _month_name(m.group(1)), _month_name(m.group(2))
        if a and b:
            tail = f"change_{a}_to_{b}"
    labels.append(tail)

    return labels, {
        "period_labels_resolved": all(not l.startswith("col_")
                                      for l in labels[:-1]),
        "month_header_found": len(mpos),
        "quarter_header_found": len(qpos),
        "year_header_found": len(cell_years),
        "tail_columns": 1,
    }


def _parse_table_a(block, kind, year, month):
    if kind == "nfp":
        if any("Major indicators of labor market activity" in l
               for l in block[:3]):
            return _parse_nfp_boxed(block, kind, year, month)
        return _parse_tab_table(block, kind, year, month)
    # ช่วงเปลี่ยนผ่าน (พ.ย. 2019 - ก.พ. 2020) ตารางยังเป็น dot leader แต่มี
    # แท็บปนอยู่ในบรรทัดข้อมูลจึงต้องเช็ค dot leader ก่อน
    n_dots = sum(1 for l in block[1:40] if re.search(r"\.{3,}", l))
    if n_dots >= 3:
        return _parse_fixed_table(block, kind, year, month)
    if any("\t" in l for l in block[1:40]):
        return _parse_tab_table(block, kind, year, month)
    return _parse_fixed_table(block, kind, year, month)


# ----------------------------------------------------------
# clean 1 ไฟล์
# ----------------------------------------------------------
def clean_bls_file(path, kind="cpi", pub_date="", source_url=""):
    path = Path(path)
    lines = _read_lines(path)
    bounds = _find_bounds(lines, kind)
    if bounds is None:
        return {"source_file": path.name, "parse_status": "title_not_found"}
    title_idx, start, end = bounds

    m = TITLE_RE[kind].search(lines[title_idx])
    month = MONTH_NUM.get(m.group(1).lower())
    year = int(m.group(2))
    if month is None:
        return {"source_file": path.name, "parse_status": "period_not_found"}
    report_period = f"{m.group(1).title()} {year}"
    doc_id = f"bls_{kind}_{year}_{month:02d}"

    if not pub_date:
        fm = re.search(r"_(\d{2})(\d{2})(\d{4})", path.stem)
        pub_date = f"{fm.group(3)}-{fm.group(1)}-{fm.group(2)}" if fm else ""
    if not source_url:
        folder = "history" if year <= 2007 else "archives"
        ext = "txt" if year <= 2007 else "htm"
        source_url = f"https://www.bls.gov/news.release/{folder}/{path.stem}.{ext}"

    # ---- หาตารางทั้งหมด + ตั้ง id ให้ไม่ซ้ำกัน ----
    found = []
    for i in range(start, end + 1):
        mt = TABLE_RE.match(lines[i])
        if mt:
            found.append({
                "line": i,
                "label": mt.group("label"),
                # "Summary table A." (ตารางสรุปหลัก) ต้องแยกจาก "Table A." ที่เป็น
                # ตารางเล่าเรื่องในเนื้อหา (เช่น "Table A. Seasonally adjusted
                # unemployment rates in 2012 and changes") ซึ่งเจอในรายงานต้นปี
                "summary": mt.group("summary").strip().lower().startswith("summary"),
                "title": lines[i].strip(),
            })
    used = {}
    for e in sorted(found, key=lambda x: not x["summary"]):
        if not e["summary"]:
            continue
        lid = e["label"]
        n_used = used.get(lid, 0)
        e["id"] = lid if n_used == 0 else f"{lid}_{n_used + 1}"
        used[lid] = n_used + 1
    rev = {}
    for e in found:
        if e["summary"]:
            continue
        lid = e["label"]
        if used.get(lid, 0) == 0:
            e["id"] = lid
        else:
            rev[lid] = rev.get(lid, 0) + 1
            e["id"] = f"{lid}_rev" if rev[lid] == 1 else f"{lid}_rev{rev[lid]}"
        used[lid] = used.get(lid, 0) + 1
    found.sort(key=lambda x: x["line"])

    spans = []
    for k, e in enumerate(found):
        s = e["line"]
        nxt = found[k + 1]["line"] if k + 1 < len(found) else end + 1
        e_end = _table_end(lines, s, nxt - 1)
        spans.append({"start": s, "end": e_end, "id": e["id"],
                      "title": e["title"], "summary": e["summary"]})

    in_table = set()
    for sp in spans:
        in_table.update(range(sp["start"], sp["end"] + 1))

    headings = [i for i in range(start, end + 1)
                if i not in in_table and _is_heading(lines, i)]

    segs = []
    if headings:
        segs.append((start, headings[0] - 1, "Overview"))
        for j, h in enumerate(headings):
            segs.append((
                h + 1,
                headings[j + 1] - 1 if j + 1 < len(headings) else end,
                lines[h].strip(),
            ))
    else:
        segs.append((start, end, "Overview"))

    def _paragraphs(idxs):
        paras, buf = [], []
        for i in idxs:
            s = lines[i].strip()
            if not s:
                if buf:
                    paras.append(re.sub(r"\s+", " ", " ".join(buf)))
                    buf = []
            else:
                buf.append(s)
        if buf:
            paras.append(re.sub(r"\s+", " ", " ".join(buf)))
        return paras

    chunks = []
    for s, e, heading in segs:
        content = "\n\n".join(
            _paragraphs([i for i in range(s, e + 1) if i not in in_table])
        )
        if not content:
            continue
        chunks.append({
            "chunk_id": f"{doc_id}_{len(chunks) + 1:03d}",
            "heading": heading,
            "content": content,
            "source_file": path.name,
            "source_url": source_url,
            "publication_date": pub_date,
            "report_period": report_period,
        })

    # ---- tables ----
    tables, table_a_status = [], {"parsed": False, "format": None}
    for sp in spans:
        block = lines[sp["start"]: sp["end"] + 1]
        entry = {
            "table_id": sp["id"],
            "title": sp["title"],
            "start_line": sp["start"] + 1,
            "end_line": sp["end"] + 1,
            "raw_text": "\n".join(block),
            "parse_status": "raw_text_only",
        }
        if sp["id"] == "A":
            parsed = _parse_table_a(block, kind, year, month)
            if parsed:
                cols, rows, notes, info = parsed
                fmt = _fmt_name(kind, block, info)
                entry.update({
                    "parse_status": "parsed_table_a",
                    "format": fmt,
                    "columns": cols,
                    "rows": rows,
                    "row_count": len(rows),
                    "notes": notes,
                    "validation": info,
                })
                table_a_status = {"parsed": True, "format": fmt}
            else:
                entry["parse_status"] = "raw_text_only"
                table_a_status = {"parsed": False, "format": _fmt_name(kind, block, {})}
        tables.append(entry)

    raw_ids = [t["table_id"] for t in tables if t["parse_status"] == "raw_text_only"]
    cleaning = {
        "header_removed": True,
        "footer_removed_from_last_modified_date": True,
        "table_a_parsed": table_a_status["parsed"],
        "table_a_format": table_a_status["format"],
        "tables_preserved_as_raw_blocks": raw_ids,
    }

    table_a_row = next((t.get("row_count") for t in tables
                        if t["table_id"] == "A"), None)
    table_a_col = next((len(t.get("columns", [])) for t in tables
                        if t["table_id"] == "A" and t.get("columns")), None)
    table_a_val = next((t
                        for t in tables if t["table_id"] == "A"), None)
    return {
        "doc_id": doc_id,
        "source": "U.S. Bureau of Labor Statistics",
        "source_file": path.name,
        "source_url": source_url,
        "publication_date": pub_date,
        "report_period": report_period,
        "title": lines[title_idx].strip(),
        "content_type": CONTENT_TYPE[kind],
        "cleaning": cleaning,
        "chunks": chunks,
        "tables": tables,
        "quality_checks": {
            "table_a_row_count": table_a_row,
            "table_a_column_count": table_a_col,
            "table_a_values_parsed": (table_a_val or {}).get("validation", {}).get("values_parsed"),
            "table_a_values_missing": (table_a_val or {}).get("validation", {}).get("values_missing"),
            "table_ids": [t["table_id"] for t in tables],
            "raw_block_table_ids": raw_ids,
            "chunk_count": len(chunks),
            "contains_last_modified_footer": any(
                FOOTER_MARKER in ln for ln in lines[start:end + 1]
            ),
        },
    }


def _fmt_name(kind, block, info):
    if kind == "nfp":
        if any("Major indicators of labor market activity" in l
               for l in block[:3]):
            return "nfp_boxed_major_indicators"
        return "nfp_summary_table_tab"
    if any(re.search(r"\.{3,}", l) for l in block[:40]):
        return "cpi_fixed_dot_leader"
    if any("\t" in l for l in block[1:40]):
        return "cpi_tab_separated"
    return "cpi_fixed_plain"


# ----------------------------------------------------------
# เขียนไฟล์ / รันทั้งโฟลเดอร์
# ----------------------------------------------------------
def write_json_atomic(obj, out_path):
    """เขียนไฟล์แบบ atomic กันไฟล์เสียหาย ถ้าถูกขัดกลางทาง"""
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fd, tmp = tempfile.mkstemp(dir=str(out_path.parent), suffix=".tmp")
    try:
        with os.fdopen(fd, "w", encoding="utf-8") as f:
            json.dump(obj, f, ensure_ascii=False, indent=2)
        os.replace(tmp, out_path)
    finally:
        if os.path.exists(tmp):
            os.remove(tmp)


def load_manifest_meta(manifest_path):
    """อ่าน publication_date / release_url จาก manifest โดย key = ชื่อไฟล์"""
    meta = {}
    if not Path(manifest_path).exists():
        return meta
    with open(manifest_path, encoding="utf-8-sig", newline="") as f:
        for row in csv.DictReader(f):
            stem = Path(row.get("local_file", "")).stem
            if not stem:
                continue
            meta[stem] = {
                "publication_date": row.get("publication_date", ""),
                "release_url": row.get("release_url", ""),
            }
    return meta


def clean_directory(text_dir, clean_dir, manifest_path, kind):
    text_dir = Path(text_dir)
    clean_dir = Path(clean_dir)
    clean_dir.mkdir(parents=True, exist_ok=True)

    meta = load_manifest_meta(manifest_path)
    files = sorted(text_dir.glob("*.txt"))
    ok, failed, summary = 0, [], []

    for path in tqdm(files, desc=f"clean {kind}", unit="file"):
        mm = meta.get(path.stem, {})
        doc = clean_bls_file(
            path, kind=kind,
            pub_date=mm.get("publication_date", ""),
            source_url=mm.get("release_url", ""),
        )
        if doc.get("parse_status") in ("title_not_found", "period_not_found"):
            failed.append((path.name, doc["parse_status"]))
            continue
        write_json_atomic(doc, clean_dir / f"{path.stem}_clean.json")
        ok += 1
        qc = doc["quality_checks"]
        ta = next((t for t in doc["tables"] if t["table_id"] == "A"), None)
        summary.append({
            "file": path.name,
            "doc_id": doc["doc_id"],
            "chunks": qc["chunk_count"],
            "tables": len(doc["tables"]),
            "table_a_status": (ta or {}).get("parse_status", "no_table_a"),
            "table_a_format": (ta or {}).get("format"),
            "table_a_rows": qc["table_a_row_count"],
            "table_a_cols": qc["table_a_column_count"],
            "values_parsed": qc["table_a_values_parsed"],
            "values_missing": qc["table_a_values_missing"],
            "footer_removed": not qc["contains_last_modified_footer"],
        })

    summary_df = pd.DataFrame(summary)
    print(f"\nสำเร็จ {ok}/{len(files)} ไฟล์ -> {clean_dir}")
    if failed:
        print(f"ล้มเหลว {len(failed)} ไฟล์:")
        for name, why in failed:
            print(f"  - {name}: {why}")
    if not summary_df.empty:
        print(f"chunk รวม: {int(summary_df['chunks'].sum()):,} chunks")
        parsed = summary_df["table_a_status"] == "parsed_table_a"
        print(f"Table A parse สำเร็จ: {int(parsed.sum())}/{len(summary_df)} ไฟล์")
        for fmt, grp in summary_df[parsed].groupby("table_a_format"):
            print(f"  - {fmt}: {len(grp)} ไฟล์ "
                  f"(แถว {int(grp['table_a_rows'].min())}-{int(grp['table_a_rows'].max())}, "
                  f"คอลัมน์ {int(grp['table_a_cols'].min())}-{int(grp['table_a_cols'].max())})")
    return summary_df


clean_summary = clean_directory(TEXT_DIR, CLEAN_DIR, MANIFEST_PATH, CLEAN_KIND)
display(clean_summary.head(10))
